In [ ]:
# 📘 cnn_lstm_trainer.ipynb

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json

# === Load Data ===
X = np.load("X.npy") / 255.0  # Normalize
y = np.load("y.npy")

# === Load Label Dictionary ===
with open("label_dict.json") as f:
    label_dict = json.load(f)
idx_to_label = {v: k for k, v in label_dict.items()}

# === Train-Test Split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === CNN + LSTM Model ===
model = models.Sequential([
    layers.TimeDistributed(layers.Conv2D(32, (3,3), activation='relu'), input_shape=(100, 128, 128, 3)),
    layers.TimeDistributed(layers.MaxPooling2D((2,2))),
    layers.TimeDistributed(layers.Conv2D(64, (3,3), activation='relu')),
    layers.TimeDistributed(layers.MaxPooling2D((2,2))),
    layers.TimeDistributed(layers.Flatten()),
    layers.LSTM(64, return_sequences=False),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(label_dict), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# === Callbacks ===
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# === Train (reduced batch size to avoid OOM) ===
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=2,
    callbacks=[early_stop],
    verbose=1
)

# === Save Model ===
model.save("cnn_lstm_model.h5")

# === Evaluate ===
y_pred = model.predict(X_test).argmax(axis=1)
print("\n📊 Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=list(label_dict.keys())))

# === Confusion Matrix ===
plt.figure(figsize=(10, 7))
sns.heatmap(
    confusion_matrix(y_test, y_pred),
    annot=True,
    fmt='d',
    xticklabels=label_dict.keys(),
    yticklabels=label_dict.keys(),
    cmap="Blues"
)
plt.title("📉 Confusion Matrix - CNN + LSTM")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_5              │ (None, 100, 126, 126,  │           896 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, 100, 63, 63,    │             0 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_7              │ (None, 100, 61, 61,    │        18,496 │
│ (TimeDistributed)               │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_8              │ (None, 100, 30, 30,    │             0 │
│ (TimeDistributed)               │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_9              │ (None, 100, 57600)     │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │    14,762,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 9)              │           585 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,786,377 (56.41 MB)

 Trainable params: 14,786,377 (56.41 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 1953s 42s/step - accuracy: 0.1800 - loss: 2.4252 - val_accuracy: 0.3043 - val_loss: 2.0449
Epoch 2/10
20/45 ━━━━━━━━━━━━━━━━━━━━ 17:33 42s/step - accuracy: 0.2329 - loss: 2.2214